# 04 — Data Quality Checks

**Week:** 6  
**Date:** September 1–7, 2026

**Goal:** Run meaningful data quality checks on the ShipTrack Silver tables and document failures and their business impact.


## Silver tables used

- `silver_scan_events`
- `silver_shipments`
- `silver_carriers`
- `silver_routes`
- `silver_hubs`

The checks below use the Silver tables already available in Databricks.


In [ ]:
silver_scan_events = spark.table("silver_scan_events")
silver_shipments = spark.table("silver_shipments")
silver_carriers = spark.table("silver_carriers")
silver_routes = spark.table("silver_routes")
silver_hubs = spark.table("silver_hubs")


## DQ-01 — Required field check

`scan_id` and `event_timestamp` are required for reliable shipment event tracking.


In [ ]:
%sql
SELECT COUNT(*) AS failed_count
FROM silver_scan_events
WHERE scan_id IS NULL
   OR event_timestamp IS NULL;

## DQ-02 — Duplicate check

`scan_id` should identify one scan event only.


In [ ]:
%sql
SELECT scan_id, COUNT(*) AS duplicate_count
FROM silver_scan_events
GROUP BY scan_id
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC;

## DQ-03 — Range check

Shipment numeric fields should not contain negative values.


In [ ]:
%sql
SELECT COUNT(*) AS failed_count
FROM silver_shipments
WHERE package_weight_kg < 0
   OR package_count < 0
   OR freight_amount_inr < 0
   OR attempt_count < 0;

## DQ-04 — Event sequence range check

`event_sequence_no` should start at 1 or higher.


In [ ]:
%sql
SELECT COUNT(*) AS failed_count
FROM silver_scan_events
WHERE event_sequence_no IS NULL
   OR event_sequence_no < 1;

## DQ-05 — Reference check

Hub, carrier, and route IDs in scan events should exist in their corresponding Silver reference tables.


In [ ]:
%sql
SELECT COUNT(*) AS failed_count
FROM silver_scan_events s
LEFT JOIN silver_hubs h
    ON s.hub_id = h.hub_id
LEFT JOIN silver_carriers c
    ON s.carrier_id = c.carrier_id
LEFT JOIN silver_routes r
    ON s.route_id = r.route_id
WHERE (s.hub_id IS NOT NULL AND h.hub_id IS NULL)
   OR (s.carrier_id IS NOT NULL AND c.carrier_id IS NULL)
   OR (s.route_id IS NOT NULL AND r.route_id IS NULL);

## DQ-06 — Timestamp order check

Shipment timestamps should follow the expected business sequence.


In [ ]:
%sql
SELECT COUNT(*) AS failed_count
FROM silver_shipments
WHERE pickup_ts < booking_ts
   OR actual_delivery_ts < pickup_ts;

## Failed record examples

The following queries capture sample failed records for evidence and review.


In [ ]:
%sql
-- DQ-01 failed examples
SELECT *
FROM silver_scan_events
WHERE scan_id IS NULL
   OR event_timestamp IS NULL
LIMIT 20;

In [ ]:
%sql
-- DQ-03 failed examples
SELECT *
FROM silver_shipments
WHERE package_weight_kg < 0
   OR package_count < 0
   OR freight_amount_inr < 0
   OR attempt_count < 0
LIMIT 20;

In [ ]:
%sql
-- DQ-04 failed examples
SELECT *
FROM silver_scan_events
WHERE event_sequence_no IS NULL
   OR event_sequence_no < 1
LIMIT 20;

In [ ]:
%sql
-- DQ-05 failed examples
SELECT
    s.scan_id,
    s.shipment_id,
    s.hub_id,
    s.carrier_id,
    s.route_id
FROM silver_scan_events s
LEFT JOIN silver_hubs h
    ON s.hub_id = h.hub_id
LEFT JOIN silver_carriers c
    ON s.carrier_id = c.carrier_id
LEFT JOIN silver_routes r
    ON s.route_id = r.route_id
WHERE (s.hub_id IS NOT NULL AND h.hub_id IS NULL)
   OR (s.carrier_id IS NOT NULL AND c.carrier_id IS NULL)
   OR (s.route_id IS NOT NULL AND r.route_id IS NULL)
LIMIT 20;

In [ ]:
%sql
-- DQ-06 failed examples
SELECT *
FROM silver_shipments
WHERE pickup_ts < booking_ts
   OR actual_delivery_ts < pickup_ts
LIMIT 20;

## Final DQ results summary

Run this cell last. The output can be used for the Week 6 evidence screenshot and the `data_quality_summary.md` file.


In [ ]:
%sql
SELECT 'DQ-01' AS rule_id, 'Required fields not null' AS rule_name, COUNT(*) AS failed_count
FROM silver_scan_events
WHERE scan_id IS NULL OR event_timestamp IS NULL

UNION ALL

SELECT 'DQ-02', 'Duplicate scan_id', COUNT(*)
FROM (
    SELECT scan_id
    FROM silver_scan_events
    GROUP BY scan_id
    HAVING COUNT(*) > 1
) d

UNION ALL

SELECT 'DQ-03', 'Shipment numeric values not negative', COUNT(*)
FROM silver_shipments
WHERE package_weight_kg < 0
   OR package_count < 0
   OR freight_amount_inr < 0
   OR attempt_count < 0

UNION ALL

SELECT 'DQ-04', 'Event sequence number >= 1', COUNT(*)
FROM silver_scan_events
WHERE event_sequence_no IS NULL OR event_sequence_no < 1

UNION ALL

SELECT 'DQ-05', 'Valid hub, carrier and route references', COUNT(*)
FROM silver_scan_events s
LEFT JOIN silver_hubs h ON s.hub_id = h.hub_id
LEFT JOIN silver_carriers c ON s.carrier_id = c.carrier_id
LEFT JOIN silver_routes r ON s.route_id = r.route_id
WHERE (s.hub_id IS NOT NULL AND h.hub_id IS NULL)
   OR (s.carrier_id IS NOT NULL AND c.carrier_id IS NULL)
   OR (s.route_id IS NOT NULL AND r.route_id IS NULL)

UNION ALL

SELECT 'DQ-06', 'Valid shipment timestamp order', COUNT(*)
FROM silver_shipments
WHERE pickup_ts < booking_ts
   OR actual_delivery_ts < pickup_ts;